# Alfvén Radius From Shell Sampling

`r_A` is defined here as the first outward radial distance where `M_A` changes from `<1` to `>1`.
This approximates the 3D `M_A=1` isosurface by directional shell/ray sampling.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from starwinds_analysis.analysis.shells import sample_spherical_shells
from starwinds_analysis.data.samples import data_file
from starwinds_analysis.physics.alfven_radius import alfven_radius_map
from starwinds_analysis.physics.alfven_radius import projected_solid_angle_weights
from starwinds_analysis.physics.alfven_radius import summarize_alfven_radius
from starwinds_analysis.smart_ds import SmartDs


In [ ]:
data_path = data_file('3d__var_4_n00000000.plt')
sds = SmartDs.from_file(data_path)
sds.prepare()
star_radius_m = float(sds('star_radius [m]'))
print(sds)
print(f'star_radius [m]: {star_radius_m:.6e}')


In [ ]:
radii = np.linspace(1.2, 20.0, 64)
shell_ds = sample_spherical_shells(
    sds,
    radii,
    fields=('M_A [none]',),
    n_polar=48,
    n_azimuth=96,
    method='nearest',
    length_unit_to_m=star_radius_m,
)

ra_map = alfven_radius_map(shell_ds)
weights = projected_solid_angle_weights(shell_ds)
polar_map = shell_ds('polar [rad]')[0]

min_ra, max_ra, avg_ra, avg_ra_cyl, coverage = summarize_alfven_radius(
    ra_map,
    polar_map,
    weights=weights,
)

print(f'Min r_A [R]: {min_ra:.3f}')
print(f'Max r_A [R]: {max_ra:.3f}')
print(f'Avg r_A [R]: {avg_ra:.3f}')
print(f'Avg cylindrical r_A [R]: {avg_ra_cyl:.3f}')
print(f'Coverage [none]: {coverage:.3f}')


In [ ]:
longitude = shell_ds('longitude [deg]')[0]
latitude = shell_ds('latitude [deg]')[0]

fig, ax = plt.subplots(figsize=(9.5, 5.2))
img = ax.pcolormesh(longitude, latitude, ra_map, shading='nearest', cmap='viridis')
ax.set_xlabel('Longitude [deg]')
ax.set_ylabel('Latitude [deg]')
ax.set_title('Alfvén Radius r_A [R]')
fig.colorbar(img, ax=ax, label='r_A [R]')
plt.show()
